# Module 07 — Explorer l'API FastAPI catalogue

API **lecture seule** sur Postgres. OpenAPI : `/docs`.

Lancer dans un terminal :
```bash
uv run presslake serve --reload
```

## Étape 1 — TestClient (sans serveur réseau)

FastAPI fournit `TestClient` pour tester l'app en mémoire.

In [1]:
from fastapi.testclient import TestClient

from presslake.api.app import create_app

client = TestClient(create_app())

r = client.get("/health")
print(r.status_code, r.json())

/home/anthony-marais/Documents/data_project/.venv/lib/python3.14/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


200 {'status': 'ok', 'service': 'presslake-catalog'}


## Étape 2 — Lister les articles

In [2]:
r = client.get("/articles", params={"limit": 3, "status": "parsed"})
data = r.json()
print("total:", data["total"])
for item in data["items"]:
    print(f"  [{item['id']}] {item['feed_id']} | {item['title'][:50] if item['title'] else '?'}…")

total: 128
  [130] hnrss | Accidental Aesthetics and Romance of Power Wires…
  [129] hnrss | Internet centralization and the original sin of NA…
  [128] hnrss | Since it was stripped of planetary status, Pluto’s…


## Étape 3 — Détail par id

In [3]:
if data["items"]:
    article_id = data["items"][0]["id"]
    r = client.get(f"/articles/{article_id}")
    article = r.json()
    print("url:", article["url"][:70])
    print("bronze:", article["s3_uri"])
    print("silver:", article.get("silver_s3_uri"))

url: https://www.governance.fyi/p/accidental-aesthetics-and-romance
bronze: s3://presslake/bronze/source=hnrss/dt=2026-08-30/abea30868212815ffc663135db59e6b408e6a2e73413bd9e800b76471f12d651.json
silver: s3://presslake/silver/source=hnrss/dt=2026-08-30/abea30868212815ffc663135db59e6b408e6a2e73413bd9e800b76471f12d651.json


## Étape 4 — Stats par statut

In [4]:
r = client.get("/stats")
stats = r.json()
print("total:", stats["total"])
for row in stats["by_status"]:
    print(f"  {row['status']:10} {row['count']}")

total: 128
  parsed     128


## Étape 5 — curl (serveur live)

```bash
curl http://127.0.0.1:8000/docs
curl "http://127.0.0.1:8000/articles?feed_id=france24&limit=2"
```

Tuto : [`docs/modules/07-fastapi.md`](../docs/modules/07-fastapi.md)